# Advanced Topics in Machine Learning: Natural Language Processing

Authors: Jonathan Bella, Alessia Berarducci, Jonas Knupp, Tobias Erbacher

Before we start, let's check whether this notebook is run on a local machine or Colab.

In [ ]:
import os
RUNNING_ON_LOCAL_MACHINE = True if 'COLAB_GPU' not in os.environ else False

import torch
GPU_AVAILABLE = torch.cuda.is_available()

assert RUNNING_ON_LOCAL_MACHINE
assert GPU_AVAILABLE

### Dataset Investigation

In [ ]:
import pandas as pd

df = pd.read_json("hf://datasets/medalpaca/medical_meadow_medical_flashcards/medical_meadow_wikidoc_medical_flashcards.json")
df = df.iloc[:, 1:]

In [ ]:
df.head()

The dataset contains 33,955 flashcards with two columns: **input** (representing the prompt or question) and **output** (representing the response or answer). These flashcards are designed to help medical students learn and review medical knowledge.

In [ ]:
# Number of documents
num_documents = len(df)
print(f"Number of flashcards: {num_documents}")

In [ ]:
# Check for missing values
print(df.isnull().sum())

In [ ]:
print("Sample Inputs:")
print(df['input'].head())

print("\nSample Outputs:")
print(df['output'].head())

In [ ]:
df['input_length'] = df['input'].apply(len)
df['output_length'] = df['output'].apply(len)

print(df[['input_length', 'output_length']].describe())

In [ ]:
duplicates_nr = df.duplicated(subset=['input', 'output']).sum()
print(f"Number of duplicate flashcards: {duplicates_nr}")

duplicates = df[df.duplicated(subset=['input', 'output'], keep=False)]

# Print out some duplicates
print("Sample duplicate flashcards:")
print(duplicates.head())

In [ ]:
# Remove duplicates
427/len(df)*100 #1.25%
#Comment: we discared around the 1.25% of the information because of duplicates.
df = df.drop_duplicates(subset=['input', 'output'])
print(f"Number of flashcards after removing duplicates: {len(df)}")

**a. Flashcard input/output Length Distribution**

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')


# Function to calculate word count
def word_count(text):
    return len(word_tokenize(str(text)))

# Calculate word counts
df['input_word_count'] = df['input'].apply(word_count)
df['output_word_count'] = df['output'].apply(word_count)

# Display statistics
print(df[['input_word_count', 'output_word_count']].describe())

In [ ]:
# Set the style
sns.set(style="whitegrid")

# Plot distribution for input lengths
plt.figure(figsize=(12, 6))
sns.histplot(df['input_word_count'], bins=30, kde=True, color='blue', label='Input')
sns.histplot(df['output_word_count'], bins=30, kde=True, color='orange', label='Output')
plt.title('Distribution of Document Lengths')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.legend()
plt.show()

**b. Vocabulary Size: number of unique words in each flashcard**

In [ ]:
# Calculate unique words
def unique_word_count(text):
    return len(set(word_tokenize(str(text).lower())))

# Calculate unique word counts
df['input_unique_words'] = df['input'].apply(unique_word_count)
df['output_unique_words'] = df['output'].apply(unique_word_count)

# Display statistics
print(df[['input_unique_words', 'output_unique_words']].describe())

In [ ]:
# Diversity of vocabulary for input/output
plt.figure(figsize=(12, 6))
sns.histplot(df['input_unique_words'], bins=30, kde=True, color='green', label='Input')
sns.histplot(df['output_unique_words'], bins=30, kde=True, color='red', label='Output')
plt.title('Distribution of Vocabulary Size per Document')
plt.xlabel('Unique Word Count')
plt.ylabel('Frequency')
plt.legend()
plt.show()

**Investigating the 0's**

In [ ]:
empty_inputs = df[df['input_unique_words'] == 0].reset_index()

In [ ]:
empty_inputs[2:20]

*Examples*

In [ ]:
empty_inputs.iloc[5]["output"]

In [ ]:
empty_inputs.iloc[6]["output"]

In [ ]:
empty_outputs = df[df['output_unique_words'] == 0]
#print("Empty Outputs:")
#print(empty_outputs)
print(empty_outputs.head())

In [ ]:
df['input_empty'] = df['input'].str.strip().eq("")
df['output_empty'] = df['output'].str.strip().eq("")
print("Empty Input Fields:", df['input_empty'].sum())
print("Empty Output Fields:", df['output_empty'].sum())

When both input and output are empty, delete rows.

In [ ]:
empty_inputs_outputs = df[(df['input_unique_words'] == 0) & (df['output_unique_words'] == 0)].reset_index()
len(empty_inputs)

402/len(df)*100 #1.18%
#Comment: we discared around the 1.18% of the information

df1 is the **new cleaned dataset** without empty fields for both input and output

In [ ]:
df1 = df[~((df['input_unique_words'] == 0) & (df['output_unique_words'] == 0))].reset_index(drop=True)

print(df1[df1['input_unique_words']==0])
len(df1[df1['input_unique_words']==0])

In [ ]:
# No empty output
print(df1[df1['output_unique_words']==0])
len(df1[df1['output_unique_words']==0])

In [ ]:
# Delete also when only input is empty
df2 = df1[~(df1['input_unique_words'] == 0)].reset_index(drop=True)

df2 is the **cleaned dataset** after removing empty cells.

In [ ]:
print(df2[df2['input_unique_words']==0])
len(df2[df2['input_unique_words']==0])

**Let's remove stopwords**

In [ ]:
from nltk.corpus import stopwords
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Define stopwords
nltk_stop_words = set(stopwords.words('english'))
custom_stop_words = ['may', 'important', 'also', 'used', 'include', 'typically']
stop_words = nltk_stop_words.union(custom_stop_words)


def preprocess(text):
    # Convert to string and remove apostrophes
    text = str(text).replace("'", "")
    # Tokenize and lowercase
    tokens = word_tokenize(text.lower())
    # Remove non-alphanumeric characters from tokens
    tokens = [re.sub(r"[^\w\s]", "", word) for word in tokens]
    # Filter out non-alphabetic tokens
    tokens = [word for word in tokens if word.isalpha()]
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    return tokens


# Apply preprocessing
df2['input_tokens'] = df2['input'].apply(preprocess)
df2['output_tokens'] = df2['output'].apply(preprocess)

# Combine all tokens from inputs and outputs
all_input_tokens = [word for tokens in df2['input_tokens'] for word in tokens]
all_output_tokens = [word for tokens in df2['output_tokens'] for word in tokens]

# Count frequency
input_freq = Counter(all_input_tokens)
output_freq = Counter(all_output_tokens)

# Get top 20
top_input = input_freq.most_common(20)
top_output = output_freq.most_common(20)

print("\nTop 20 Most Common Words in Inputs (Excluding Stopwords):")
print(top_input)

print("\nTop 20 Most Common Words in Outputs (Excluding Stopwords):")
print(top_output)

# Visualization of Top Words

# Inputs
input_top_df2 = pd.DataFrame(top_input, columns=['Word', 'Frequency'])
plt.figure(figsize=(12, 8))
sns.barplot(data=input_top_df2, x='Frequency', y='Word', palette='Blues_d')
plt.title('Top 20 Most Common Words in Inputs (Excluding Stopwords)')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.show()

# Outputs
output_top_df2 = pd.DataFrame(top_output, columns=['Word', 'Frequency'])
plt.figure(figsize=(12, 8))
sns.barplot(data=output_top_df2, x='Frequency', y='Word', palette='Reds_d')
plt.title('Top 20 Most Common Words in Outputs (Excluding Stopwords)')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate the number of unique tokens in inputs and outputs
unique_input_tokens = set(all_input_tokens)
unique_output_tokens = set(all_output_tokens)

# Count the unique tokens
num_unique_input_tokens = len(unique_input_tokens)
num_unique_output_tokens = len(unique_output_tokens)

print(f"Number of unique tokens in inputs: {num_unique_input_tokens}")
print(f"Number of unique tokens in outputs: {num_unique_output_tokens}")


*We've created a customized stopwords list* to handle specific cases (e.g. may, also...)

In [ ]:
df2.head()

In [ ]:
def compute_vocab_growth_unique_tokens(token_series):
    vocab = set()
    vocab_sizes = []
    for tokens in token_series:
        vocab.update(tokens)  # Add unique tokens from each document
        vocab_sizes.append(len(vocab))  # Track vocabulary size at this point
    return vocab_sizes

# Vocabulary growth for inputs and outputs
input_vocab_growth = compute_vocab_growth_unique_tokens(df2['input_tokens'])
output_vocab_growth = compute_vocab_growth_unique_tokens(df2['output_tokens'])

# Plot vocabulary growth
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(input_vocab_growth) + 1), input_vocab_growth, label='Input Vocabulary Growth')
plt.plot(range(1, len(output_vocab_growth) + 1), output_vocab_growth, label='Output Vocabulary Growth')
plt.xlabel("Number of Documents")
plt.ylabel("Vocabulary Size (Unique Words)")
plt.title("Vocabulary Growth Curve (Using Unique Words)")
plt.legend()
plt.tight_layout()
plt.show()

- Toward the end (around 30,000 documents), the growth of the lines slows down slightly. This indicates that most common words have already been encountered, and adding more documents introduces fewer new unique words.


- The vocabulary in the output is significantly larger than that in the input.

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Generate word clouds
def generate_wordcloud(tokens_series, title):
    # Flatten the list of tokenized words into a single string
    text = " ".join(" ".join(tokens) for tokens in tokens_series)
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

    plt.figure(figsize=(10, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(title, fontsize=16)
    plt.show()

# Word clouds for input and output
generate_wordcloud(df2['input_tokens'], "Word Cloud for Input")
generate_wordcloud(df2['output_tokens'], "Word Cloud for Output")

### Using spacy
- en_core_web_sm

In [ ]:
import spacy
from gensim.models import Word2Vec
import pandas as pd

# English model
nlp = spacy.load("en_core_web_sm")

# Preprocess text using spaCy
def preprocess_with_spacy(text):
    doc = nlp(str(text).lower())  # Process the text and lowercase it
    tokens = [
        token.text for token in doc
        if token.is_alpha and not token.is_stop  # Keep only alphabetic tokens and exclude stopwords
    ]
    return tokens


df2['question_tokens_spacy'] = df2['input'].apply(preprocess_with_spacy)
df2['answer_tokens_spacy'] = df2['output'].apply(preprocess_with_spacy)

# Combine token input and output for Word2Vec training
all_sentences = df2['question_tokens_spacy'].tolist() + df2['answer_tokens_spacy'].tolist()

# Train Word2Vec model
word2vec_model = Word2Vec(
    sentences=all_sentences,  # Tokenized sentences
    vector_size=100,          # Size of word embeddings
    window=5,                 # Context window size
    min_count=2,              # Minimum frequency of words
    workers=4,                # Number of CPU threads
    sg=0                      # Use CBOW (sg=0) or Skip-Gram (sg=1)
)

word2vec_model.save("word2vec_model_medicine_spacy.bin")

# Vocabulary size
print("Vocabulary size:", len(word2vec_model.wv))

# Test the model with a word
example_word = 'disease'
if example_word in word2vec_model.wv:
    print("Most similar words to '{}':".format(example_word))
    print(word2vec_model.wv.most_similar(example_word))
else:
    print("Word '{}' not in vocabulary.".format(example_word))

In [ ]:
# Count unique words
unique_input_words = set([word for tokens in df2['question_tokens_spacy'] for word in tokens])
unique_output_words = set([word for tokens in df2['answer_tokens_spacy'] for word in tokens])

print(f"Number of unique words in inputs (spaCy): {len(unique_input_words)}")
print(f"Number of unique words in outputs (spaCy): {len(unique_output_words)}")


- Including Lemmatization

In [ ]:
# Preprocess text with lemmatization
def preprocess_with_spacy_lemmatization(text):
    doc = nlp(str(text).lower())  # Process the text and lowercase it
    tokens = [
        token.lemma_ for token in doc
        if token.is_alpha and not token.is_stop  # Keep only alphabetic tokens and exclude stopwords
    ]
    return tokens


# Apply the lemmatization-based preprocessing
df2['question_tokens_spacy_lemma'] = df2['input'].apply(preprocess_with_spacy_lemmatization)
df2['answer_tokens_spacy_lemma'] = df2['output'].apply(preprocess_with_spacy_lemmatization)

# Count unique words after lemmatization
unique_input_words_lemma = set([word for tokens in df2['question_tokens_spacy_lemma'] for word in tokens])
unique_output_words_lemma = set([word for tokens in df2['answer_tokens_spacy_lemma'] for word in tokens])

print(f"Number of unique words in inputs after lemmatization: {len(unique_input_words_lemma)}")
print(f"Number of unique words in outputs after lemmatization: {len(unique_output_words_lemma)}")

In [ ]:
# Combine lemmatized token input and output for Word2Vec training
all_sentences_lemma = df2['question_tokens_spacy_lemma'].tolist() + df2['answer_tokens_spacy_lemma'].tolist()

# Train Word2Vec model on lemmatized tokens
word2vec_model_lemma = Word2Vec(
    sentences=all_sentences_lemma,  # Tokenized sentences (lemmatized)
    vector_size=100,                # Size of word embeddings
    window=5,                       # Context window size
    min_count=2,                    # Minimum frequency of words
    workers=4,                      # Number of CPU threads
    sg=0                            # Use CBOW (sg=0) or Skip-Gram (sg=1)
)

# Save the model
word2vec_model_lemma.save("word2vec_model_medicine_spacy_lemma.bin")

# Vocabulary size
print("Vocabulary size after lemmatization:", len(word2vec_model_lemma.wv))

# Test the model with a word
example_word = 'disease'
if example_word in word2vec_model_lemma.wv:
    print("Most similar words to '{}' (lemmatized):".format(example_word))
    print(word2vec_model_lemma.wv.most_similar(example_word))
else:
    print("Word '{}' not in vocabulary (lemmatized).".format(example_word))

- SpaCy for medicine "en_core_sci_md"

In [ ]:
import spacy
from gensim.models import Word2Vec
import pandas as pd

# English model
nlp = spacy.load("en_core_sci_md")

# Preprocess text using spaCy
def preprocess_with_spacy(text):
    doc = nlp(str(text).lower())  # Process the text and lowercase it
    tokens = [
        token.text for token in doc
        if token.is_alpha and not token.is_stop  # Keep only alphabetic tokens and exclude stopwords
    ]
    return tokens


df2['question_tokens_spacy'] = df2['input'].apply(preprocess_with_spacy)
df2['answer_tokens_spacy'] = df2['output'].apply(preprocess_with_spacy)

# Combine token input and output for Word2Vec training
all_sentences = df2['question_tokens_spacy'].tolist() + df2['answer_tokens_spacy'].tolist()

# Train Word2Vec model
word2vec_model = Word2Vec(
    sentences=all_sentences,  # Tokenized sentences
    vector_size=100,          # Size of word embeddings
    window=5,                 # Context window size
    min_count=2,              # Minimum frequency of words
    workers=4,                # Number of CPU threads
    sg=0                      # Use CBOW (sg=0) or Skip-Gram (sg=1)
)


word2vec_model.save("word2vec_model_medicine_spacy.bin")

# Vocabulary size
print("Vocabulary size:", len(word2vec_model.wv))

# Test the model with a word
example_word = 'disease'
if example_word in word2vec_model.wv:
    print("Most similar words to '{}':".format(example_word))
    print(word2vec_model.wv.most_similar(example_word))
else:
    print("Word '{}' not in vocabulary.".format(example_word))

In [ ]:
# Count unique words
unique_input_words = set([word for tokens in df2['question_tokens_spacy3'] for word in tokens])
unique_output_words = set([word for tokens in df2['answer_tokens_spacy3'] for word in tokens])

print(f"Number of unique words in inputs (spaCy): {len(unique_input_words)}")
print(f"Number of unique words in outputs (spaCy): {len(unique_output_words)}")

Examples

In [ ]:
example_word = 'ibd'
if example_word in word2vec_model.wv:
    print("Most similar words to '{}':".format(example_word))
    print(word2vec_model.wv.most_similar(example_word))
else:
    print("Word '{}' not in vocabulary.".format(example_word))

In [ ]:
example_word = 'heart'
if example_word in word2vec_model.wv:
    print("Most similar words to '{}':".format(example_word))
    print(word2vec_model.wv.most_similar(example_word))
else:
    print("Word '{}' not in vocabulary.".format(example_word))

*Meanings of some abbreviations:*

- Graft versus Host Diseas gvhd

- Certified Anesthesiologist Assistants (CAAs)

- kimmelstiel kidney condition associated with long-standing diabetes.

- ibd, Inflammatory Bowel Disease (IBD)

- horseshoe, a horseshoe kidney is a congenital condition that causes the kidneys to join and form a horseshoe shape.

**Nearest Neighbors for Multiple Words**

In [ ]:
for word in ['disease', 'treatment', 'patient']:
    if word in word2vec_model.wv:
        print(f"Most similar words to '{word}':")
        print(word2vec_model.wv.most_similar(word))

**Outlier Detection: Find the word that doesn't belong in a group.**

In [ ]:
print(word2vec_model.wv.doesnt_match(['heart', 'kidney', 'lung']))

### Vector embeddings using openAI

In [ ]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # Loads environment variables from the .env file

API_TOKEN_AI = os.getenv("API_TOKEN_AI")
print(API_TOKEN_AI)  # Outputs your token (or None if not found)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from google.colab import files

#uploaded = files.upload()

openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    raise ValueError("OpenAI API key not found. Please set it in the .env file.")

In [ ]:
df2.head()

**Another way**

In [ ]:
from openai import OpenAI


client = OpenAI(api_key=API_TOKEN_AI)

df4 = pd.DataFrame(columns=['combined','ada_embedding'])

#We are using tokes from spacy
df4['combined'] = df2.apply(
    lambda row: ' '.join(row['question_tokens_spacy']) + ' ' + ' '.join(row['answer_tokens_spacy']),
    axis=1
)

# Verify the DataFrame
print(df4)
print(type(df4))

In [ ]:
import pandas as pd
import time
from tqdm import tqdm  # For progress bar
import numpy as np
from ast import literal_eval


# This function fetches the embedding for a given text using the specified model.
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

#  Attempts to fetch the embedding with retries in case of rate limit errors.
def get_embedding_with_retry(text, model="text-embedding-3-small", max_retries=5, backoff_factor=2):
    for attempt in range(1, max_retries + 1):
        try:
            return get_embedding(text, model)
        except Exception as e:
            print(f"Attempt {attempt} failed with error: {e}")
            if attempt == max_retries:
                print("Max retries reached. Skipping this text.")
                return None
            sleep_time = backoff_factor ** attempt
            print(f"Retrying in {sleep_time} seconds...")
            time.sleep(sleep_time)


# df4 = pd.read_csv('/path/to/your/input.csv')

embeddings = []

# Define the delay between requests (in seconds)
# To stay under 50 RPS, delay should be at least 0.02 seconds
# Adding a slight buffer for safety
DELAY_BETWEEN_REQUESTS = 0.025  # 25 ms

# Iterate over the 'combined' column with a progress bar
for text in tqdm(df4['combined'], desc="Generating Embeddings"):
    embedding = get_embedding_with_retry(text, model='text-embedding-3-small')
    embeddings.append(embedding)
    time.sleep(DELAY_BETWEEN_REQUESTS)  # Respect rate limit


df4['ada_embedding'] = embeddings

df4.to_csv('/content/embedded_flashcards.csv', index=False)
print("Embeddings generated and saved successfully.")

We could have used more powerful models, but the text is only in English. It doesn't require such complexity. The model that we've used is 'text-embedding-3-small.'

In [ ]:
# Load the embeddings
import pandas as pd

df4= pd.read_csv('/content/embedded_flashcards (1).csv')
df4['ada_embedding'] = df4.ada_embedding.apply(eval).apply(np.array)

In [ ]:
print(df4)

In [ ]:
import pandas as pd
from sklearn.manifold import TSNE
import numpy as np
from ast import literal_eval

# Convert the 'ada_embedding' to an array
matrix = np.array(df4['ada_embedding'].tolist())


# Create a t-SNE model and transform the data
tsne = TSNE(n_components=2, perplexity=40, random_state=42, init='random', learning_rate=200)
vis_dims = tsne.fit_transform(matrix)
vis_dims.shape
print(tsne)

In [ ]:
df4.head()

- K Means, If you want to cluster in the original high-dimensional space (matrix) and prioritize accuracy.

- Perform clustering in the original high-dimensional space (matrix) to get more reliable clusters.
Use the 2D t-SNE visualization (vis_dims) for exploratory or visual analysis but avoid using it for clustering directly.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Determine inertia for different values of k
inertia = []
k_values = range(1, 11)  # Test k from 1 to 10
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(matrix)
    inertia.append(kmeans.inertia_)

# Plot the Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker='o')
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.xticks(k_values)
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# Perform k-means clustering
kmeans = KMeans(n_clusters=5, random_state=42)
df4['cluster'] = kmeans.fit_predict(matrix)

# Add t-SNE results to the DataFrame
df4['tsne_x'] = vis_dims[:, 0]
df4['tsne_y'] = vis_dims[:, 1]



# Visualize clusters
plt.figure(figsize=(10, 6))
plt.scatter(df4['tsne_x'], df4['tsne_y'], c=df4['cluster'], cmap='viridis', alpha=0.6)
plt.title("t-SNE Visualization with K-Means Clusters")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
# Group the data by cluster and inspect its content
for cluster in df4['cluster'].unique():
    print(f"Cluster {cluster}:")
    print(df4[df4['cluster'] == cluster]['combined'].head(10))  # View top 10 items in the cluster
    print("\n")

In [ ]:
# Add cluster results to df4
df4['cluster'] = kmeans.labels_

# Group items by clusters and save to clustered_results
clustered_results = df4.groupby('cluster')['combined'].apply(list).reset_index()


clustered_results.columns = ['Cluster', 'Items']
print(clustered_results)

**Analyze top cluster results**

In [ ]:
# Save top 10 items per cluster in a new DataFrame
top_items_per_cluster = (
    df4.groupby('cluster')['combined']
    .apply(lambda x: x.head(10).tolist())
    .reset_index()
)


top_items_per_cluster.columns = ['Cluster', 'Top_Items']

print(top_items_per_cluster)


clustered_results.to_csv('full_clustered_results.csv', index=False) # Full result
top_items_per_cluster.to_csv('top_items_per_cluster.csv', index=False) #Only 10 results for cluster

In [ ]:
print(top_items_per_cluster['Top_Items'].head())
print(top_items_per_cluster['Top_Items'].apply(type).value_counts())

In [ ]:
cluster_text = top_items_per_cluster['Top_Items'].apply(lambda x: " ".join(x))


from sklearn.feature_extraction.text import TfidfVectorizer

# Apply TF-IDF to extract keywords
vectorizer = TfidfVectorizer(stop_words='english', max_features=10)  # Limit to 10 top keywords
tfidf_matrix = vectorizer.fit_transform(cluster_text)

# Extract the keywords for each cluster
keywords = []
for row in tfidf_matrix.toarray():
    top_indices = row.argsort()[-10:][::-1]  # Get indices of the top 10 keywords
    keywords.append(" ".join(vectorizer.get_feature_names_out()[top_indices]))

# Add the keywords to the DataFrame
top_items_per_cluster['Keywords'] = keywords

# Display the updated DataFrame
print(top_items_per_cluster[['Cluster', 'Keywords']])
top_items_per_cluster.to_csv('cluster_keywords.csv', index=False)

- "Calcium Regulation"
- "Sleep Disorders"
- "Biochemical Regulation"
- "Metabolism and Glucose Regulation"
- "Antibiotics and Pharmacology"

**3dimensional**

In [ ]:
from sklearn.manifold import TSNE

# Create a t-SNE model with 3 components
tsne = TSNE(
    n_components=3,        # Change to 3 for 3D
    perplexity=40,
    random_state=42,
    init='random',
    learning_rate=200
)

# Transform the data
vis_dims = tsne.fit_transform(matrix)
print("t-SNE Shape:", vis_dims.shape)  # Should output (num_samples, 3)

In [ ]:
import pandas as pd

# Assuming df4 is your DataFrame and 'ada_embedding' is the embedding column
df4[['tsne_x', 'tsne_y', 'tsne_z']] = pd.DataFrame(vis_dims, index=df4.index)

In [ ]:
from sklearn.cluster import KMeans

# Perform k-means clustering on the original embeddings
kmeans = KMeans(n_clusters=15, random_state=42)
df4['cluster'] = kmeans.fit_predict(matrix)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Necessary for 3D plotting

# Create a 3D scatter plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot
scatter = ax.scatter(
    df4['tsne_x'],
    df4['tsne_y'],
    df4['tsne_z'],
    c=df4['cluster'],
    cmap='viridis',
    alpha=0.6
)

# Add titles and labels
ax.set_title("t-SNE 3D Visualization with K-Means Clusters")
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.set_zlabel("t-SNE Dimension 3")

# Add a color bar
cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label("Cluster")

plt.show()

In [ ]:
for i, row in df4.iterrows():
    plt.text(row['tsne_x'], row['tsne_y'], str(i), fontsize=9)


plt.scatter(
    df4['tsne_x'],
    df4['tsne_y'],
    c=df4['cluster'],
    cmap='viridis',
    alpha=0.6,
    s=50,  # Size of points
    marker='o'  # Shape of points
)

In [ ]:
import plotly.express as px

fig = px.scatter(
    df4,
    x='tsne_x',
    y='tsne_y',
    color='cluster',
    title="t-SNE Visualization with K-Means Clusters",
    opacity=0.6
)
fig.show()

In [ ]:
import plotly.express as px

# Create a 3D scatter plot
fig = px.scatter_3d(
    df4,
    x='tsne_x',  # Dimension 1
    y='tsne_y',  # Dimension 2
    z='tsne_z',  # Dimension 3
    color='cluster',  # Coloring by cluster
    title="t-SNE 3D Visualization with K-Means Clusters",
    opacity=0.7  # Set point transparency
)

# Show the interactive plot
fig.show()

Cluster 0: : Relationship between low levels of certain factors or substances. Likely involves medical or biochemical pathways where "low levels" are a significant characteristic.

Cluster 1: These items may represent topics related to sleep studies, neurology, or mental health issues.

Cluster 2: Regulation of δ-aminolevulinic acid and substances involved in biochemical regulation. Likely connected to metabolic or enzymatic functions.

Cluster 3: Issues involving glucose, high C-peptide levels, and insulin. Likely tied to diabetes, insulin resistance, or endocrine disorders.

Cluster 4: β-lactams, penicillin, cephalosporins, and inhibition mechanisms. This cluster seems to focus on antibiotics and bacterial resistance mechanisms.

In [ ]:
from scipy.spatial.distance import cdist
import pandas as pd

# Assuming df_clusters contains the t-SNE coordinates and cluster labels
# Compute centroids of each cluster in the 3D space
centroids = df4.groupby('cluster')[['tsne_x', 'tsne_y', 'tsne_z']].mean()

# Compute pairwise distances between centroids
distances = cdist(centroids, centroids, metric='euclidean')

# Convert distances to a DataFrame for better readability
distance_df = pd.DataFrame(
    distances,
    index=centroids.index,
    columns=centroids.index
)

# Display the distances
print(distance_df)

In [ ]:
df4.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Extract items from clusters as text data
cluster_items = df4.groupby('cluster')['combined'].apply(' '.join)

# Use CountVectorizer to find common terms across clusters
vectorizer = CountVectorizer(stop_words='english', max_features=20)
X = vectorizer.fit_transform(cluster_items)

# Create a DataFrame of terms and their frequencies per cluster
terms = vectorizer.get_feature_names_out()
term_frequencies = pd.DataFrame(X.toarray(), index=cluster_items.index, columns=terms)

# Find clusters with overlapping terms
overlap_matrix = term_frequencies.T.corr()

# Display the thematic overlap matrix
print(overlap_matrix)

- Cluster 1 and 3 share a strong thematic similarity.

- Cluster 0 and 1 share a moderate thematic similarity.

- Cluster 1 and Cluster 2 have an inverse relationship in their themes.

- Cluster 4 has low or negative correlations with all other clusters, implying it represents a unique and distinct theme.

- Cluster 0 and Cluster 4 (-0.036) show almost no overlap, indicating unrelated topics.

In [ ]:
# Extract terms and their frequencies for Clusters 1 and 3
cluster_1_terms = term_frequencies.loc[1]
cluster_3_terms = term_frequencies.loc[3]

# Combine terms into a DataFrame for comparison
comparison_df = pd.DataFrame({
    'Cluster 1 Frequency': cluster_1_terms,
    'Cluster 3 Frequency': cluster_3_terms
}).sort_values(by=['Cluster 1 Frequency', 'Cluster 3 Frequency'], ascending=False)

# Add a column to highlight shared terms (where both clusters have non-zero frequency)
comparison_df['Shared Term'] = (comparison_df['Cluster 1 Frequency'] > 0) & (comparison_df['Cluster 3 Frequency'] > 0)

# Display the DataFrame
print(comparison_df)

### Indexing and Searching

**WHOOSH (Indexing and Searching)**

Whoosh is a pure Python search engine library that allows you to index and search text data efficiently. It uses the TF-IDF (Term Frequency-Inverse Document Frequency) algorithm by default to rank search results based on their relevance.

Whoosh handles tokenization internally based on the schema you define. Whoosh will tokenize, analyze, and process the text according to the field definitions in the schema.

In [ ]:
!pip install pandas whoosh

In [ ]:
flashcards_df=df
# Check if 'id' column exists; if not, create one
if 'id' not in flashcards_df.columns:
    flashcards_df.insert(0, 'id', range(1, len(flashcards_df) + 1))

# Verify the IDs
print(flashcards_df.head())

In [ ]:
import pandas as pd
from whoosh import index
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StemmingAnalyzer
from whoosh.qparser import MultifieldParser
import os
import shutil



# Generate unique IDs (if not already present)
if 'id' not in flashcards_df.columns:
    # Method 1: Incremental Integer IDs
    flashcards_df.insert(0, 'id', range(1, len(flashcards_df) + 1))



# Check validate data
required_columns = {'id', 'input', 'output'}
if not required_columns.issubset(flashcards_df.columns):
    missing = required_columns - set(flashcards_df.columns)
    raise ValueError(f"The following required columns are missing from the dataset: {missing}")

# Drop rows with missing values in required columns
flashcards_df = flashcards_df.dropna(subset=['id', 'input', 'output'])

# 4. Convert DataFrame to list of dictionaries
flashcards = flashcards_df.to_dict('records')

# 5. Define the Whoosh schema
schema = Schema(
    id=ID(stored=True, unique=True),
    input=TEXT(stored=True, analyzer=StemmingAnalyzer()),
    output=TEXT(stored=True, analyzer=StemmingAnalyzer())
)

# Create the index
index_dir = "flashcard_index"

# If the index directory exists, delete it to prevent schema mismatches
if os.path.exists(index_dir):
    shutil.rmtree(index_dir)
    print(f"Deleted existing index directory '{index_dir}' due to schema mismatch.")

# Create a new index directory
os.mkdir(index_dir)
ix = index.create_in(index_dir, schema)
print(f"Created new index with the updated schema in '{index_dir}'.")

# Index the flashcards
with ix.writer() as writer:
    for card in flashcards:
        try:
            writer.update_document(
                id=str(card["id"]),       # Ensure 'id' is a string
                input=card["input"],
                output=card["output"]
            )
        except KeyError as e:
            print(f"Missing key {e} in flashcard: {card}")
    # No need to call writer.commit() explicitly as 'with' handles it
    print("Indexing completed.")

# Define the search function
def search_flashcards(query_str, limit=10):
    with ix.searcher() as searcher:
        parser = MultifieldParser(["input", "output"], schema=ix.schema)
        try:
            query = parser.parse(query_str)
        except Exception as e:
            print(f"Error parsing query: {e}")
            return
        results = searcher.search(query, limit=limit)
        if not results:
            print("No matching flashcards found.")
            return
        for hit in results:
            print(f"ID: {hit['id']}")
            print(f"Question: {hit['input']}")
            print(f"Answer: {hit['output']}")
            print("-" * 40)

# Interactive search
if __name__ == "__main__":
    print("Flashcard Search")
    print("Type 'exit' to quit.")
    while True:
        user_query = input("Enter search query: ")
        if user_query.lower() == 'exit':
            print("Exiting search.")
            break
        search_flashcards(user_query)

---

### Model Training and Evaluation

@Jonas, @Jonatan

---

### Speech-to-Text and Text-to-Speech

Let us first import the libraries necessary to run this notebook. However, the audio libraries (recording, playback) that can be used in Colab or locally differ, and some of the TTS-models cannot be run on CPU-only systems. So we will programmatically check in which environment this notebook is run.

In [ ]:
!pip install openai-whisper

if RUNNING_ON_LOCAL_MACHINE:
  !pip install sounddevice pynput pyttsx3
  import pyttsx3 # Model 1
  from pynput import keyboard
else:
  !apt-get install -y portaudio19-dev
  !pip install sounddevice ffmpeg-python
  !sudo apt-get install texlive texlive-latex-extra dvipng cm-super -y
  from IPython.display import HTML, Audio, display
  from google.colab.output import eval_js
  from base64 import b64decode
  from scipy.io.wavfile import read as wav_read
  import io
  import ffmpeg
  import ipywidgets as widgets
  from google.colab import output

import math
import scipy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import time
import sounddevice as sd
import threading

# Speech-to-Text
import librosa
from scipy.io.wavfile import write
import queue
import whisper

# Text-to-Speech
if GPU_AVAILABLE:
  !pip install inflect unidecode
  import unidecode # Model 2
  import inflect

##### Speech-to-Text

In [ ]:
MAX_DURATION = 120                              # Maximum audio recording duration in seconds.
SAVE_AUDIO = False                              # True: Audio will be saved after recording; False: Audio will not be saved after recording.
OUTPUT_FILENAME = "speech_input.wav"            # Name of the audio file that will be saved. Remember to give it a filetype, i.e. ".wav" at the end.
PLAY_AFTER_RECORDING = False                    # Do you wish to replay the audio after the recording?

In [ ]:
SAMPLING_RATE = 16000                           # Hz. Note that openai whisper requires 160000 Hz
NUMBER_OF_RECORDING_CHANNELS = 1                # Number of channels (1 for mono, 2 for stereo). Note that openai whisper requires mono channel.

WAVEFORM_PARAMS = {                             # Plot parameters for the wave form.
    'figure.figsize': (16, 6),
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'grid.alpha': 0.5,
    'legend.fontsize': 12,
    #'legend.frameon': False,
    'axes.grid': True,
    'axes.labelsize': 12,
    'lines.linewidth': 0.5,
    # Font settings for LaTeX
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern'],
}

MEL_SPECTROGRAM_WINDOW_SIZE = 0.025             # Mel spectrogram window size in seconds.
MEL_SPECTROGRAM_HOP_SIZE = 0.01                 # Mel spectrogram hop size (delay between consecutive windows) in seconds.
MEL_SPECTROGRAM_START_TIME = 0                  # Mel spectrogram start time in seconds.
N_FFT = 1300                                    # Mel spectrogram sample number for Fast Fourier Transform.
N_MEL = 80                                      # Mel spectrogram band number to generate.

MEL_SPECTROGRAM_PARAMS = {                      # Plot parameters for the wave form.
    'figure.figsize': (16, 6),
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'grid.alpha': 0.5,
    'legend.fontsize': 12,
    #'legend.frameon': False,
    'axes.grid': True,
    'axes.labelsize': 12,
    'lines.linewidth': 0.5,
    # Font settings for LaTeX
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern'],
}

In [ ]:
# Only relevant for non-local environments.
AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_p = document.createElement("P");
var my_btn = document.createElement("BUTTON");
var t = document.createTextNode("Press to start recording");

my_btn.appendChild(t);
//my_p.appendChild(my_btn);
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0;
var reader;
var recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
  gumStream = stream;
  var options = {
    //bitsPerSecond: 8000, //chrome seems to ignore, always 48k
    mimeType : 'audio/webm;codecs=opus'
    //mimeType : 'audio/webm;codecs=pcm'
  };
  //recorder = new MediaRecorder(stream, options);
  recorder = new MediaRecorder(stream);
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true;
    preview.src = url;
    document.body.appendChild(preview);

    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() {
      base64data = reader.result;
      //console.log("Inside FileReader:" + base64data);
    }
  };
  recorder.start();
  };

recordButton.innerText = "Recording... press to stop";

navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);


function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!"
  }
}

// https://stackoverflow.com/a/951057
function sleep(ms) {
  return new Promise(resolve => setTimeout(resolve, ms));
}

var data = new Promise(resolve=>{
//recordButton.addEventListener("click", toggleRecording);
recordButton.onclick = ()=>{
toggleRecording()

sleep(2000).then(() => {
  // wait 2000ms for the data to be available...
  // ideally this should use something like await...
  //console.log("Inside data:" + base64data)
  resolve(base64data.toString())

});

}
});

</script>
"""

First we need to set up the audio recorder. We do this with an InputStream and a buffer. Execute the cell and speak. To stop the recording, press the space bar.

In [ ]:
def listenToMyQuestion():
  audio_recording_queue = queue.Queue()
  stop_recording = False

  audio_recording_buffer = []

  def audio_callback(input_data, frames, time, status): # Do not remove the not needed parameters as they are somehow used in the InputStream
    if status:
      print(status)
    audio_recording_queue.put(input_data.copy())

  AUDIO_RECORDING_START_TIME = time.time()

  if RUNNING_ON_LOCAL_MACHINE:
    def on_press(key):
      global stop_recording
      if key == keyboard.Key.space:
        stop_recording = True
        key_listener.stop()
        print("Recording has been stopped by the user.")

    with sd.InputStream(samplerate=SAMPLING_RATE, channels=NUMBER_OF_RECORDING_CHANNELS, callback=audio_callback):
      key_listener = keyboard.Listener(on_press=on_press)
      key_listener.start()

      while not stop_recording:
        if time.time() - AUDIO_RECORDING_START_TIME < MAX_DURATION:
          audio_recording_buffer.append(audio_recording_queue.get())
        else:
          print("Recording has been stopped due to maximum length.")
          key_listener.stop()
          break
    AUDIO_RECORDING_FORMATTED = np.concatenate(audio_recording_buffer, axis=0).squeeze()
    VOICE_INPUT = AUDIO_RECORDING_FORMATTED / np.max(np.abs(AUDIO_RECORDING_FORMATTED))
  else:
    # THIS IS ADAPTED FROM THE TA's NOTEBOOK "4_Speech.ipynb"
    def audio_recorder():
      global SAMPLING_RATE
      display(HTML(AUDIO_HTML))
      RECORDING = eval_js("data")
      BINARY_RECORDING = b64decode(RECORDING.split(',')[1])

      process = (ffmpeg
        .input('pipe:0')
        .output('pipe:1', format='wav', ar=SAMPLING_RATE)
        .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True)
      )
      output, err = process.communicate(input=BINARY_RECORDING)

      RIFF_CHUNK_SIZE = len(output) - 8
      Q = RIFF_CHUNK_SIZE
      b = []
      for i in range(4):
        Q, R = divmod(Q, 256)
        b.append(R)

      RIFF = output[:4] + bytes(b) + output[8:]

      AUDIO_DATA = io.BytesIO(RIFF)
      SAMPLING_RATE, AUDIO = wav_read(AUDIO_DATA)
      VOICE_INPUT = torch.tensor(AUDIO / np.max(np.abs(AUDIO)), dtype=torch.float32)

      return VOICE_INPUT, SAMPLING_RATE

    VOICE_INPUT, SAMPLING_RATE = audio_recorder()
  
  if SAVE_AUDIO:
    write(OUTPUT_FILENAME, SAMPLING_RATE, (VOICE_INPUT * 32767).astype(np.int16))

  if PLAY_AFTER_RECORDING and not RUNNING_ON_LOCAL_MACHINE:
    Audio(VOICE_INPUT, rate=SAMPLING_RATE)
  
  return VOICE_INPUT, SAMPLING_RATE

Now that we have recorded the audio, we can put it into a speech-to-text model of our choice. We will use the openai-whisper base-model:

In [ ]:
if GPU_AVAILABLE:
  model = whisper.load_model("base").to("cuda")
else:
  model = whisper.load_model("base")

Next, we can detect what language was spoken:

In [ ]:
def languageDetection(voice_input):
  _, LANGUAGE_PROBABILITIES = model.detect_language(whisper.log_mel_spectrogram(whisper.pad_or_trim(voice_input)).to(device=model.device))
  DETECTED_LANGUAGE = max(LANGUAGE_PROBABILITIES, key=LANGUAGE_PROBABILITIES.get)
  print("Detected language: " + str(DETECTED_LANGUAGE) + ", confidence: " + str(round(LANGUAGE_PROBABILITIES[DETECTED_LANGUAGE] * 100, 2)))

Thereupon, we transform the audio to text.

In [ ]:
def speechToText(voice_input):
  return model.transcribe(voice_input)["text"]

##### Speech-to-Text Analytics

If you wish to perform some analytics on the recorded voice input, you can run the following cells. First, you can take a look at the wave form.

In [ ]:
def waveform(voice_input, sampling_rate):
  DURATION = voice_input.shape[0] / sampling_rate
  TIME = np.linspace(0, DURATION, voice_input.shape[0])

  mpl.rcParams.update(WAVEFORM_PARAMS)
  plt.plot(TIME, voice_input)
  plt.title("Recorded Audio Waveform")
  plt.xlabel("time [s]")
  plt.ylabel("Amplitude [a.u.]")
  plt.show()

Next up, we can plot the Mel-spectrogram:

In [ ]:
def melSpectrogram(voice_input, sampling_rate):
  W = int(math.ceil(MEL_SPECTROGRAM_WINDOW_SIZE * SAMPLING_RATE))
  D = int(math.ceil(MEL_SPECTROGRAM_HOP_SIZE * SAMPLING_RATE))
  T = int(math.ceil(MEL_SPECTROGRAM_START_TIME * SAMPLING_RATE))
  DURATION = voice_input.shape[0] / sampling_rate

  if not RUNNING_ON_LOCAL_MACHINE:
    VOICE_INPUT = voice_input.numpy()

  MEL_SPECTROGRAM = librosa.feature.melspectrogram(y=VOICE_INPUT, sr=SAMPLING_RATE, n_fft=N_FFT, win_length=W, hop_length=D, n_mels=N_MEL).squeeze()

  MEL_SPECTROGRAM_FREQUENCY_BAND_BIN_EDGES = np.arange(MEL_SPECTROGRAM.shape[0] + 1)
  MEL_SPECTROGRAM_TIME_SLICES = np.linspace(0, DURATION, MEL_SPECTROGRAM.shape[1] + 1)
  MEL_SPECTROGRAM_POWER = librosa.power_to_db(MEL_SPECTROGRAM, ref=np.max)

  MEL_FREQUENCIES = np.append(librosa.mel_frequencies(n_mels=N_MEL, fmax=SAMPLING_RATE / 2), SAMPLING_RATE / 2)

  mpl.rcParams.update(MEL_SPECTROGRAM_PARAMS)
  plt.pcolormesh(MEL_SPECTROGRAM_TIME_SLICES, MEL_SPECTROGRAM_FREQUENCY_BAND_BIN_EDGES, MEL_SPECTROGRAM_POWER)
  plt.colorbar(label='Power [dB]')
  plt.title("Mel Spectrogram of Recorded Audio")
  plt.xlabel('Time [s]')
  plt.ylabel('Frequency [Hz]')
  plt.yticks(ticks=MEL_SPECTROGRAM_FREQUENCY_BAND_BIN_EDGES[9::10], labels=[f'{freq:.0f}' for freq in MEL_FREQUENCIES[9::10]])
  plt.show()

##### Text-to-Speech

There are two different models in use here. One is for local-machines using the os-delivered text-to-speech functionalities, the other one is for server-machines like Colab making use of Tacotron2 and WaveGlow.

In [ ]:
#local-model parameters
SHOW_AVAILABLE_VOICES = False
PYTTSX3_VOICE = 132
PYTTSX3_TALKING_SPEED = 150
PYTTSX3_VOLUME = 1

#server-model parameters
WAVEGLOW_SAMPLING_RATE = 22050

In [ ]:
if GPU_AVAILABLE:
  TACOTRON2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2', model_math='fp32').to('cuda')

  WAVEGLOW = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', model_math='fp32')
  WAVEGLOW = WAVEGLOW.remove_weightnorm(WAVEGLOW).to('cuda')

  TTS_UTILS = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tts_utils')
else:
  print("Error: Cannot run the Tacotron2 model without GPU.")

In [ ]:
def textToSpeech(text):
  if RUNNING_ON_LOCAL_MACHINE:
    def pyttsx3_speak(engine, text, event):
      VOICES = engine.getProperty('voices')
      if SHOW_AVAILABLE_VOICES:
        for idx, voice in enumerate(VOICES):
          print("Voice: " + str(idx) + ", ID: " + str(voice.id) + ", Name: " + str(voice.name) + ", Language: " + str(voice.languages))
      engine.setProperty('voice', VOICES[PYTTSX3_VOICE].id)
      engine.setProperty('rate', PYTTSX3_TALKING_SPEED)
      engine.setProperty('volume', PYTTSX3_VOLUME)
      try:
        if engine._inLoop:
          engine.endLoop()
        engine.say(text)
        event.set()
        engine.runAndWait()
      except:
        engine.stop()
        engine.endLoop()
        print("Response interrupted.")

    def pyttsx3_initEngine(callback, text, event):
      global PYTTSX3_ENGINE
      PYTTSX3_ENGINE = pyttsx3.init()
      print("Playing system response...")
      callback(PYTTSX3_ENGINE, text, event)

    def pyttsx3_speakThread(text):
      thread_finished_event = threading.Event()
      thread = threading.Thread(target=pyttsx3_initEngine, args=(pyttsx3_speak, text, thread_finished_event))
      thread.start()
      thread_finished_event.wait()

    print("Waiting for system response...")
    pyttsx3_speakThread(text)
    print("System response generated.\n")
  else:
    if GPU_AVAILABLE:
      WAVEGLOW_SEQUENCES, WAVEGLOW_LENGTHS = TTS_UTILS.prepare_input_sequence([TEXT])
      with torch.no_grad():
        TACOTRON2_MEL, _, _ = TACOTRON2.infer(WAVEGLOW_SEQUENCES, WAVEGLOW_LENGTHS)
        TACOTRON2_VOICE = WAVEGLOW.infer(TACOTRON2_MEL)[0].data.cpu().numpy()
    Audio(TACOTRON2_VOICE, rate=WAVEGLOW_SAMPLING_RATE)
  print("System response: " + text)

---

### Interactive Section

Here we can interact with our model by asking it a question.

In [ ]:
VOICE_INPUT, SAMPLING_RATE = listenToMyQuestion()
languageDetection(VOICE_INPUT)
TEXT = speechToText(VOICE_INPUT)

The system receives the following prompt:

In [ ]:
TEXT

For a plot of the waveform and Mel-spectrogram, run the subsequent two cells:

In [ ]:
waveform(VOICE_INPUT, SAMPLING_RATE)

In [ ]:
melSpectrogram(VOICE_INPUT, SAMPLING_RATE)

Now we pass the prompt to the system:

@Jonas, Jonatan

In [ ]:
# ...
SYSTEM_OUTPUT = "..."

Lastly, we generate the voice output:

In [ ]:
textToSpeech(SYSTEM_OUTPUT)